# 07: KG Evolution

Rule-based KG completion: adds `viennakg:inferredTag` facts to POIs that
no source CSV states directly, derived purely from triples already present in the KG. 
Obviously these are basic rules, but should be sufficient to showcase the idea of KG completion and inference. 

1. **FamilyFriendly**: Park with Playground AND Water feature amenities true.
2. **ActivePlayground**: PlaygroundArea with 4+ distinct equipment types true.
3. **QuietPocketPark**: Park under 3000 sqm without a Playground amenity.
4. **OldTownLandmark**: TouristAttraction in Bezirk1 (Innere Stadt).
5. **TransitAccessibleForFamilies**: PlaygroundArea within 300m of a Stop.

In [1]:
import sys
sys.path.insert(0, "..")

from rdflib import Graph

from kg.enrichment.infer_tags import infer_tags, RULES
from reasoning.gtfs_routing import GtfsRouter
from reasoning.preference_filter import find_pois, describe_poi, _candidate_pois

## 1. Load the KG before tagging

In [2]:
g = Graph()
g.parse("../kg/vienna_mobility_kg.ttl", format="turtle")
before = len(g)
print(f"Loaded {before} triples")

tagged_already = list(g.query('''
PREFIX viennakg: <http://example.org/viennakg#>
SELECT (COUNT(*) as ?c) WHERE { ?poi viennakg:inferredTag ?t . }
'''))[0].c
print(f"inferredTag triples present before running the rules: {tagged_already}")

Loaded 80485 triples
inferredTag triples present before running the rules: 0


## 2. Run the completion rules

`infer_tags(g)` runs all 5 rules and adds a `viennakg:inferredTag` triple
for every match. Working on an in-memory copy here (not overwriting the
real KG file from the notebook), the actual pipeline step is
`python kg/enrichment/infer_tags.py`, run once after `build_kg.py`.

In [3]:
counts = infer_tags(g)
after = len(g)
print(f"\n{after} triples now ({after - before} new), "
      f"{sum(counts.values())} inferredTag matches across all 5 rules")

  FamilyFriendly                    465 POIs tagged
  ActivePlayground                  491 POIs tagged
  QuietPocketPark                   313 POIs tagged
  OldTownLandmark                    87 POIs tagged
  TransitAccessibleForFamilies      679 POIs tagged

82520 triples now (2035 new), 2035 inferredTag matches across all 5 rules


## 3. Idempotency check

Rerunning the same rules on the same graph should add nothing new as this is standard KG completion behavior especially with the library rdflib.Graph and the g.add() method that we us.

In [4]:
before_rerun = len(g)
counts_rerun = infer_tags(g, verbose=False)
after_rerun = len(g)
print(f"Before rerun: {before_rerun} triples. After rerun: {after_rerun} triples "
      f"({after_rerun - before_rerun} new).")
assert after_rerun == before_rerun, "rerun should be a no-op"
print("Confirmed idempotent.")

Before rerun: 82520 triples. After rerun: 82520 triples (0 new).
Confirmed idempotent.


## 4. Spot-check one POI per rule

`describe_poi()` now lists inferred tags alongside the usual structured fields.

In [5]:
for tag, rule in RULES.items():
    matches = sorted(rule(g))
    example = matches[0]
    print(f"=== {tag} ({len(matches)} matches) ===")
    print(describe_poi(g, example))
    print()

=== FamilyFriendly (465 matches) ===
in 21. Floridsdorf -- 5,288 m² -- amenities: Playground, Water feature -- tags: FamilyFriendly

=== ActivePlayground (491 matches) ===
in 21. Floridsdorf -- amenities: Klettern, Rutschen, Sand-Matsch, Schaukeln, Wasserspiel, Wippen -- tags: ActivePlayground, TransitAccessibleForFamilies

=== QuietPocketPark (313 matches) ===
in 22. Donaustadt -- 1,099 m² -- tags: QuietPocketPark

=== OldTownLandmark (87 matches) ===
Sehenswürdigkeit -- in 1. Innere Stadt -- at Herrengasse 6-8 -- tags: OldTownLandmark

=== TransitAccessibleForFamilies (679 matches) ===
in 21. Floridsdorf -- amenities: Klettern, Rutschen, Sand-Matsch, Schaukeln, Wasserspiel, Wippen -- tags: ActivePlayground, TransitAccessibleForFamilies



## 5. Using inferred tags as filters

In [ ]:
family_friendly_parks = _candidate_pois(g, poi_classes=["Park"], required_amenities=["FamilyFriendly"])
print(f"Parks matching required_amenities=['FamilyFriendly']: {len(family_friendly_parks)}")
print("(should equal the FamilyFriendly rule's own match count above)") # it is ;)

Parks matching required_amenities=['FamilyFriendly']: 465
(should equal the FamilyFriendly rule's own match count above)


In [7]:
router = GtfsRouter(date="20260815")

# Karlsplatz
origin_lon, origin_lat = 16.368948, 48.200955

results = find_pois(g, router, origin_lon, origin_lat,
                     poi_classes=["Park"], required_amenities=["FamilyFriendly"],
                     top_n=3)
for r in results:
    print(f"{r['name']}: {r['travel_time_min']} min" if r["reachable"] else f"{r['name']}: no connection found")
    print(f"  {r['description']}")

Resselpark: 2.9 min
  in 4. Wieden -- 41,531 m² -- amenities: Dogs allowed, Playground, Water feature -- tags: FamilyFriendly
Stadtpark - Kinderpark: 3.5 min
  in 3. Landstraße -- 35,826 m² -- amenities: Playground, Water feature -- tags: FamilyFriendly
Josef-Pfeifer-Park: 6.5 min
  in 3. Landstraße -- 625 m² -- amenities: Playground, Water feature -- tags: FamilyFriendly


## 6. Persist the tagged graph

This is what `python kg/enrichment/infer_tags.py` does on its own actually but if you run it from the notebook
here too so `kg/vienna_mobility_kg.ttl` actually carries the inferred tags going forward.

In [8]:
g.serialize(destination="../kg/vienna_mobility_kg.ttl", format="turtle")
print(f"Saved {len(g)} triples to kg/vienna_mobility_kg.ttl")

Saved 82520 triples to kg/vienna_mobility_kg.ttl
